# Chapter 3 | Early Identification of At-Risk Students using Logistic Regression

**Reproducible companion notebook**  
*Higher education · Academic risk classification · Interpretable predictive modelling*

This notebook follows the methods and figures presented in Chapter 3 of the accompanying book. It covers risk-label construction, socio-academic exploratory analysis, logistic regression, model evaluation, tiered support and group-level fairness diagnostics. Figures **3.2–3.14** are generated in the same order as the book and saved in PDF and PNG formats.

**Data source:** Nevriye Yilmaz and Boran Şekeroğlu (2019), *Higher Education Students Performance Evaluation*, UCI Machine Learning Repository. DOI: [10.24432/C51G82](https://doi.org/10.24432/C51G82). Dataset: [UCI repository](https://archive.ics.uci.edu/dataset/856/higher+education+students+performance+evaluation). Licence: **CC BY 4.0**.

**Interpretation:** The outcome is defined from course grades; predictions are illustrative decision-support signals, not determinations about individual students. Some dataset variables are categorical codes, and the small sample limits generalisation. The model and plots below retain the supplied chapter implementation rather than substituting a different modelling approach.

## 1. Environment and data preparation

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Libraries and reproducibility</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Import the analysis tools and apply a consistent, publication-oriented chart style.</span>
</div>

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
SLATE = '#5D6D7E'
CORAL = '#EC7063'
INK = '#2C3E50'
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True, parents=True)

sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams.update({'axes.titlesize': 13, 'axes.labelsize': 12,
                     'xtick.labelsize': 10, 'ytick.labelsize': 10,
                     'figure.dpi': 110, 'savefig.dpi': 300})

RISK_NAMES = {0: 'Not At Risk (0)', 1: 'At Risk (1)'}

def save_figure(fig, file_stem):
    for extension in ('pdf', 'png'):
        fig.savefig(FIG_DIR / f'{file_stem}.{extension}',
                    bbox_inches='tight', facecolor='white')
    plt.show()

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Load the UCI higher-education dataset</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Use DATA (1).csv if available; otherwise download the official archive and cache the source CSV.</span>
</div>

In [ ]:
UCI_ARCHIVE = (
    'https://archive.ics.uci.edu/static/public/856/'
    'higher%2Beducation%2Bstudents%2Bperformance%2Bevaluation.zip'
)
LOCAL_FILES = [Path('DATA (1).csv'), Path('data/DATA (1).csv'),
               Path('datasets/DATA (1).csv')]


def locate_csv_in_archive(archive_bytes):
    with ZipFile(BytesIO(archive_bytes)) as archive:
        candidates = [name for name in archive.namelist()
                      if name.lower().endswith('.csv') and not name.startswith('__MACOSX/')]
        if not candidates:
            raise FileNotFoundError('No CSV file was found in the UCI archive.')
        preferred = next((n for n in candidates if Path(n).name == 'DATA (1).csv'),
                         candidates[0])
        return archive.read(preferred)


local_file = next((p for p in LOCAL_FILES if p.is_file()), None)
if local_file is not None:
    df = pd.read_csv(local_file)
    print(f'Loaded local dataset: {local_file}')
else:
    try:
        req = Request(UCI_ARCHIVE, headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(req, timeout=30) as response:
            csv_bytes = locate_csv_in_archive(response.read())
    except Exception as exc:
        raise FileNotFoundError(
            'Could not retrieve the dataset. Download the official UCI archive, '
            'extract DATA (1).csv, and place it beside this notebook or in data/. '
            'Dataset: https://archive.ics.uci.edu/dataset/856/'
            'higher+education+students+performance+evaluation'
        ) from exc
    data_cache = Path('data')
    data_cache.mkdir(exist_ok=True, parents=True)
    (data_cache / 'DATA (1).csv').write_bytes(csv_bytes)
    df = pd.read_csv(BytesIO(csv_bytes))
    print('Downloaded official UCI data and cached data/DATA (1).csv')

print('Dataset shape:', df.shape)
display(df.head())

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Validate grades and construct academic risk</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Define At Risk as grade codes 0, 1 or 2; retain all remaining grades as Not At Risk.</span>
</div>

In [ ]:
df.columns = df.columns.astype(str).str.strip()
required = {'STUDENT ID', 'GRADE', '1', '2', '4', '11', '17', '22', '29'}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f'The dataset is missing expected columns: {missing}')

df['GRADE'] = pd.to_numeric(df['GRADE'], errors='raise')
if df['GRADE'].isna().any() or not df['GRADE'].isin(range(8)).all():
    raise ValueError('Expected complete course grades encoded as integers 0–7.')

# Grades 0=Fail, 1=DD and 2=DC are defined as the risk category.
df['Risk'] = (df['GRADE'] <= 2).astype(int)

print('Grade codes:', sorted(df['GRADE'].unique().tolist()))
print('Risk counts:')
display(df['Risk'].map(RISK_NAMES).value_counts().rename('Students').to_frame())
print('Risk proportion:', f"{df['Risk'].mean():.1%}")
print('Missing feature values:', int(df.drop(columns=['Risk']).isna().sum().sum()))

## 2. Exploratory analysis of socio-academic factors

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Shared grouped-count chart</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Define one plotting function for consistent, comparable risk distributions across the selected educational variables.</span>
</div>

In [ ]:
def risk_count_plot(feature, categories, title, xlabel, file_stem, figsize=(8, 4.8)):
    counts = pd.crosstab(df[feature], df['Risk']).reindex(
        index=categories, columns=[0, 1], fill_value=0
    )
    x = np.arange(len(categories))
    fig, ax = plt.subplots(figsize=figsize)
    width = 0.39
    ax.bar(x - width/2, counts[0].values, width=width,
           label=RISK_NAMES[0], color=SLATE)
    ax.bar(x + width/2, counts[1].values, width=width,
           label=RISK_NAMES[1], color=CORAL)
    ax.set_xticks(x, [str(v) for v in categories])
    ax.set(title=title, xlabel=xlabel, ylabel='Number of students')
    ax.legend(frameon=False, title='Risk status')
    sns.despine(ax=ax)
    fig.tight_layout()
    save_figure(fig, file_stem)
    return counts

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.2 · Scholarship Type vs Academic Risk</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Compare risk categories across the five scholarship levels.</span>
</div>

In [ ]:
risk_count_plot('4', list(range(1, 6)),
                'Scholarship Type vs Academic Risk',
                'Scholarship (1=None, 2=25%, 3=50%, 4=75%, 5=Full)',
                'figure_3_2_scholarship_vs_risk')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.3 · Weekly Study Hours vs Academic Risk</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Examine the association between study-time categories and observed academic risk.</span>
</div>

In [ ]:
risk_count_plot('17', list(range(1, 6)),
                'Weekly Study Hours vs Academic Risk',
                'Weekly study hours (1=None, 2=<5, 3=6–10, 4=11–20, 5=>20)',
                'figure_3_3_study_hours_vs_risk')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.4 · Class Attendance vs Academic Risk</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Compare risk counts for different reported class-attendance patterns.</span>
</div>

In [ ]:
risk_count_plot('22', list(range(1, 4)),
                'Class Attendance vs Academic Risk',
                'Attendance (1=Always, 2=Sometimes, 3=Never)',
                'figure_3_4_attendance_vs_risk')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.5 · Prior Semester GPA vs Academic Risk</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Inspect how previous academic performance relates to the current course risk label.</span>
</div>

In [ ]:
risk_count_plot('29', list(range(1, 6)),
                'Prior Semester GPA vs Academic Risk',
                'Prior GPA category (1=<2.00, 2=2.00–2.49, 3=2.50–2.99, 4=3.00–3.49, 5=>3.49)',
                'figure_3_5_prior_gpa_vs_risk')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.6 · Mother's Education vs Academic Risk</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Describe the risk distribution across maternal education categories without assuming causality.</span>
</div>

In [ ]:
risk_count_plot('11', list(range(1, 7)),
                "Mother's Education vs Academic Risk",
                "Mother's education (1=Primary … 6=PhD)",
                'figure_3_6_mother_edu_vs_risk')

## 3. Logistic regression and predictive evaluation

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Select predictors and prepare the train–test split</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Exclude the identifier and outcome, stratify the 75:25 split, and fit the scaler on training observations only.</span>
</div>

In [ ]:
# Preserve the original chapter predictors and modelling specification.
# 'STUDENT ID' identifies records; 'GRADE' defines the target and must be excluded.
X = df.drop(columns=['STUDENT ID', 'GRADE', 'Risk']).copy()
y = df['Risk'].copy()

for column in X.columns:
    X[column] = pd.to_numeric(X[column], errors='raise')
if X.isna().any().any():
    raise ValueError('All model predictors must be complete.')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Predictors:', len(X.columns))
print('Training cases:', len(X_train), '| Test cases:', len(X_test))
print('Training risk proportion:', f'{y_train.mean():.1%}')
print('Test risk proportion:', f'{y_test.mean():.1%}')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Fit the logistic regression classifier</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Train the balanced logistic model and compute test-set predictions, risk probabilities and evaluation metrics.</span>
</div>

In [ ]:
model = LogisticRegression(
    max_iter=2000, solver='liblinear', class_weight='balanced',
    random_state=RANDOM_STATE
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
auc_value = roc_auc_score(y_test, y_proba)

print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=[RISK_NAMES[0], RISK_NAMES[1]],
    digits=3, zero_division=0
))
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'Test ROC-AUC: {auc_value:.3f}')
print('Confusion matrix [TN, FP; FN, TP]:')
print(cm)

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.7 · Confusion matrix</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Display correct classifications and false positives/negatives on the held-out test subset.</span>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 5.6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            linewidths=.6, linecolor='white', ax=ax, annot_kws={'size': 14})
ax.set(title='Confusion Matrix (Logistic Regression)',
       xlabel='Predicted label', ylabel='Observed label')
ax.set_xticklabels([RISK_NAMES[0], RISK_NAMES[1]], rotation=0)
ax.set_yticklabels([RISK_NAMES[0], RISK_NAMES[1]], rotation=0)
fig.tight_layout()
save_figure(fig, 'figure_3_7_confusion_matrix')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.8 · ROC curve</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Inspect discrimination across classification thresholds, summarised by the area under the curve.</span>
</div>

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(7.0, 5.4))
ax.plot(fpr, tpr, color=SLATE, linewidth=2.2,
        label=f'Logistic regression (AUC = {auc_value:.3f})')
ax.plot([0, 1], [0, 1], color=INK, linestyle='--', linewidth=1,
        label='Chance reference')
ax.set(xlim=(0, 1), ylim=(0, 1.02), title='Receiver Operating Characteristic',
       xlabel='False positive rate', ylabel='True positive rate')
ax.legend(loc='lower right', frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'figure_3_8_roc_curve')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.9 · Predicted risk probabilities</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Compare predicted probabilities for observed risk classes; overlapping distributions indicate uncertainty.</span>
</div>

In [ ]:
prob_df = pd.DataFrame({
    'Predicted risk probability': y_proba,
    'Observed status': pd.Series(y_test.to_numpy()).map(RISK_NAMES)
})
fig, ax = plt.subplots(figsize=(8.6, 5.2))
sns.histplot(data=prob_df, x='Predicted risk probability',
             hue='Observed status', hue_order=[RISK_NAMES[0], RISK_NAMES[1]],
             palette={RISK_NAMES[0]: SLATE, RISK_NAMES[1]: CORAL},
             bins=12, stat='density', common_norm=False,
             element='step', alpha=.38, ax=ax)
ax.set(title='Predicted Risk Probability Distribution by Observed Class',
       xlim=(0, 1), ylabel='Density')
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'figure_3_9_predicted_probability_distribution')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.10 · Standardised coefficients</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Identify the largest fitted coefficients; encoded category values must not be read as causal effects.</span>
</div>

In [ ]:
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_.ravel(),
})
coef_df['absolute_coefficient'] = coef_df['coefficient'].abs()
top_coef = coef_df.nlargest(12, 'absolute_coefficient').sort_values('coefficient')
fig, ax = plt.subplots(figsize=(9.3, 6.1))
ax.barh(top_coef['feature'], top_coef['coefficient'],
        color=[CORAL if v > 0 else SLATE for v in top_coef['coefficient']])
ax.axvline(0, linestyle='--', linewidth=1, color=INK)
ax.set(title='Largest Standardised Logistic Regression Coefficients',
       xlabel='Coefficient (standardised model inputs)', ylabel='Dataset feature code')
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'figure_3_10_logreg_coefficients')
display(coef_df.sort_values('absolute_coefficient', ascending=False).head(12))

**Modelling note.** To remain consistent with the supplied Chapter 3 analysis, ordinal-coded dataset variables enter the model as standardised numeric columns, and the course identifier remains a predictor. This is a reproducible baseline, not a preferred encoding for every application: category distances and coefficient interpretation depend on the source coding. Comparisons with one-hot encoding, exclusion of course identifiers and external validation are appropriate extensions. The reported test metrics describe this split only.

## 4. Risk-based support tiers and educational interpretation

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Define illustrative support tiers</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Apply the chapter’s 0.40 and 0.70 probability boundaries; these are illustrative, not empirically optimised cut-offs.</span>
</div>

In [ ]:
results_test = X_test.copy()
results_test['true_risk'] = y_test.to_numpy()
results_test['pred_risk'] = y_pred
results_test['pred_prob_risk'] = y_proba

TIER_CUTPOINTS = (0.40, 0.70)
TIER_ORDER = ['Low Support', 'Targeted Support', 'Intensive Support']
results_test['support_tier'] = pd.cut(
    results_test['pred_prob_risk'], bins=[-np.inf, *TIER_CUTPOINTS, np.inf],
    labels=TIER_ORDER, include_lowest=True
)

tier_summary = (
    results_test.groupby('support_tier', observed=False)
    .agg(students=('true_risk', 'size'),
         mean_predicted_risk=('pred_prob_risk', 'mean'),
         observed_risk_rate=('true_risk', 'mean'))
    .reindex(TIER_ORDER)
)
display(tier_summary)

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.11 · Distribution of support tiers</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Summarise the number of test cases assigned to each illustrative level of academic support.</span>
</div>

In [ ]:
tier_counts = results_test['support_tier'].value_counts().reindex(TIER_ORDER, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 5.0))
bars = ax.bar(TIER_ORDER, tier_counts.values, color=[SLATE, '#96A9B9', CORAL], width=.57)
for bar in bars:
    ax.annotate(f'{bar.get_height():.0f}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points', ha='center')
ax.set(title='Intervention Tiers Derived from Predicted Risk Probabilities',
       ylabel='Number of students', xlabel='Support tier')
ax.set_ylim(0, max(1, tier_counts.max()) * 1.18)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'figure_3_11_support_tier_distribution')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.12 · Risk probability boundaries</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Visualise the relationship between estimated risk and the chapter’s illustrative support-tier thresholds.</span>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.1))
sns.histplot(results_test['pred_prob_risk'], bins=12, color=SLATE, ax=ax)
for value in TIER_CUTPOINTS:
    ax.axvline(value, color=INK, linewidth=1.6, linestyle='--')
ymax = ax.get_ylim()[1]
for x, label in [(0.20, 'Low'), (0.55, 'Targeted'), (0.85, 'Intensive')]:
    ax.text(x, ymax * .92, label, ha='center', va='top', color=INK, fontsize=11)
ax.set(title='Predicted Risk Probability and Illustrative Support Boundaries',
       xlim=(0, 1), xlabel='Predicted probability of At Risk (class 1)',
       ylabel='Number of students')
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 'figure_3_12_risk_probability_thresholds')

## 5. Group-level fairness diagnostics

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Calculate group-level rates</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Compute selection, true-positive and false-positive rates, retaining undefined rates as missing in small groups.</span>
</div>

In [ ]:
def group_metrics(results, group_col, group_labels):
    rows = []
    for code_value, sample in results.groupby(group_col):
        observed = sample['true_risk'].to_numpy()
        predicted = sample['pred_risk'].to_numpy()
        tp = np.sum((observed == 1) & (predicted == 1))
        tn = np.sum((observed == 0) & (predicted == 0))
        fp = np.sum((observed == 0) & (predicted == 1))
        fn = np.sum((observed == 1) & (predicted == 0))
        rows.append({
            'group': group_labels.get(int(code_value), str(code_value)),
            'n': len(sample),
            'selection_rate': np.mean(predicted == 1),
            'TPR': tp / (tp + fn) if tp + fn else np.nan,
            'FPR': fp / (fp + tn) if fp + tn else np.nan,
            'positives': int(tp + fn), 'negatives': int(fp + tn),
        })
    return pd.DataFrame(rows)

SEX_LABELS = {1: 'Female', 2: 'Male'}
AGE_LABELS = {1: '18–21', 2: '22–25', 3: 'Above 26'}
fair_sex = group_metrics(results_test, '2', SEX_LABELS)
fair_age = group_metrics(results_test, '1', AGE_LABELS)
print('Diagnostics by sex:')
display(fair_sex)
print('Diagnostics by age group:')
display(fair_age)

def fairness_chart(group_table, group_order, colors, title, filename):
    metric_columns = ['selection_rate', 'TPR', 'FPR']
    rows = group_table.set_index('group').reindex(group_order)
    fig, ax = plt.subplots(figsize=(9.0, 5.3))
    x = np.arange(len(metric_columns))
    width = .75 / len(group_order)
    for i, group_name in enumerate(group_order):
        vals = rows.loc[group_name, metric_columns].to_numpy(dtype=float)
        xpos = x + (i - (len(group_order)-1)/2) * width
        ax.bar(xpos, vals, width=width * .92, color=colors[i], label=group_name)
    ax.set_xticks(x, ['Selection rate', 'True positive rate', 'False positive rate'])
    ax.set(ylim=(0, 1.03), title=title, ylabel='Rate')
    ax.legend(title='Group', frameon=False)
    sns.despine(ax=ax)
    fig.tight_layout()
    save_figure(fig, filename)

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.13 · Fairness diagnostics by sex</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Compare group-level prediction rates while considering sample sizes and possible instability.</span>
</div>

In [ ]:
fairness_chart(fair_sex, ['Female', 'Male'], [SLATE, CORAL],
               'Group-Level Fairness Diagnostics by Sex',
               'figure_3_13_fairness_by_sex')

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Figure 3.14 · Fairness diagnostics by age group</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Examine age-related differences without interpreting observed disparities as proof of bias or its absence.</span>
</div>

In [ ]:
fairness_chart(fair_age, ['18–21', '22–25', 'Above 26'],
               [SLATE, '#92A6B8', CORAL],
               'Group-Level Fairness Diagnostics by Age Group',
               'figure_3_14_fairness_by_age')

## 6. Interpretation and reproducibility checks

The empirical workflow demonstrates a classification-based screening method rather than an autonomous intervention policy. False negatives may represent missed opportunities for support, while false positives may trigger unnecessary referrals. Decisions about assistance require contextual review, and support tiers should be calibrated to institutional needs and resources.

**Comparison with the book.** The printed Chapter 3 reports a test confusion matrix of **TN=13, FP=7, FN=5, TP=12** and **ROC-AUC=0.738**. The check below displays any differences rather than modifying the computed values. Exact reproduction depends on the original dataset, feature coding, library versions and modelling settings. No results are hard-coded into the graphs.

<div style="border-left: 4px solid #476579; background: #f4f7f9; padding: 10px 14px; border-radius: 6px; margin: 14px 0 7px;">
<strong style="color: #263f52;">Compare computed outputs with the printed chapter</strong><br>
<span style="color: #4b5964; font-size: 0.94em;">Report any discrepancies between the re-run and the book’s reference figures without replacing model-derived results.</span>
</div>

In [ ]:
book_cm = np.array([[13, 7], [5, 12]])
book_auc = 0.738
comparison = pd.DataFrame({
    'Measure': ['True negatives', 'False positives', 'False negatives',
                'True positives', 'ROC-AUC'],
    'Printed in book': [13, 7, 5, 12, book_auc],
    'Computed in notebook': [cm[0,0], cm[0,1], cm[1,0], cm[1,1], auc_value],
})
comparison['Difference'] = (comparison['Computed in notebook'] - comparison['Printed in book']).round(4)
display(comparison)
print('Generated figures:', len(list(FIG_DIR.glob('figure_3_*.pdf'))), 'PDF files')

### Dataset citation and licence

Yilmaz, N., & Şekeroğlu, B. (2019). *Higher Education Students Performance Evaluation* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C51G82. Dataset licence: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

The notebook retrieves the dataset from its source or reads a separately downloaded copy. If the dataset is redistributed alongside future materials, retain the source attribution and licence notice. All charts are generated from the data loaded during execution.